# ENG-209, Partie 2, Série 7

## Introduction à la regression linéaire avec Numpy et Scikit-Learn

Le but des exercices suivants est de vous familiariser avec les concepts de la régression linéaire, en abordant d’abord les fondements mathématiques, puis en introduisant le package de data science Scikit-Learn.

## Compréhension de la donnée

Pour ces exercices, nous utiliserons les résultats d’une [enquête fictive](https://www.scribbr.com/statistics/multiple-linear-regression/) menée dans 500 villes, visant à collecter des données sur le pourcentage de fumeurs, le pourcentage de personnes se rendant au travail à vélo, ainsi que le pourcentage de personnes ayant déclaré des problèmes cardiaques.

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from numpy.typing import NDArray

In [ ]:
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size']      = 16

In [ ]:
data: pd.DataFrame = pd.read_csv("../data/heart/heartData.csv")

### Exercice

Rappel: avant d’appliquer vos modèles de régression, prenez **un instant pour valider** que vous comprenez bien la structure du jeu de données fourni.

Sans faire une analyse exploratoire complète, vérifiez simplement que les données correspondent à la description fournie et qu’elles sont exploitables.

**Questions à considérer :**

- Quelle est la taille du jeu de données (nombre de lignes, de variables) ?

- Les variables correspondent-elles à la description annoncée (type, sens, cohérence générale) ?

- Voyez-vous des valeurs manquantes ou manifestement erronées (e.g. pourcentage négatif, ou supérieur à 100% de la population) ?

- Les statistiques de base confirment-elles vos attentes (ordres de grandeur, distributions, etc.) ?

In [ ]:
data.info()

In [ ]:
data.describe()

**Réponse**:

498 mesures correspondant à différentes villes, comprenant les trois variables (pourcentages) décrites. Il n’y a pas de valeurs manquantes et les données semblent cohérentes.

**Exercice – Visualisation rapide de la donnée**

Réalisez quelques visualisations simples pour mieux comprendre la relation entre les variables explicatives et la variable à prédire.
L’objectif est de repérer visuellement d’éventuelles tendances ou corrélations, sans faire une analyse détaillée.

Veillez à ce que les axes soient clairement étiquetés et que les unités soient indiquées.

**Questions à considérer** :

- Observez-vous des relations linéaires ou non linéaires entre la variable cible et les autres variables ?

- Certaines variables semblent-elles plus fortement corrélées avec la variable à prédire ?

- Y a-t-il des groupes (clusters), tendances ou valeurs atypiques visibles sur les graphiques ?

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.scatter(
    x=data.biking,
    y=data['heart.disease'],
    marker='.',
    label="Mesures",
)
ax.grid(True)
ax.legend(loc='best', fontsize=10)
ax.set_title("Vélo au travail vs santé cardiaque\nMesures en % de la population dans chaque ville", fontsize=12)
ax.set_xlabel("Vélo au travail (% population)", fontsize=10)
ax.set_ylabel("Maladies cardiaques (% population)", fontsize=10)
ax.tick_params(axis='both', labelsize=10) 
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.scatter(
    data.smoking,
    data['heart.disease'],
    marker='.',
    label="Mesures",
)
ax.grid(True)
ax.legend(loc='best', fontsize=10)
ax.set_title("Tabagisme et santé cardiaque\nMesures en % de la population dans chaque ville", fontsize=12)
ax.set_xlabel("Fumeurs (% de la population)", fontsize=10)
ax.set_ylabel("Maladies cardiaques (% population)", fontsize=10)
ax.tick_params(axis='both', labelsize=10)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.scatter(
    x=data['biking'],
    y=data['smoking'],
    marker='.',
    label="Mesures",
)

ax.grid(True)
ax.legend(loc='best', fontsize=10)
ax.set_title("Vélo et tabagisme", fontsize=12)
ax.set_xlabel("Travail à vélo (% de la population)", fontsize=10)
ax.set_ylabel("Fumeurs (% de la population)", fontsize=10)
ax.tick_params(axis='both', labelsize=10) 
plt.show()

In [ ]:
# Rappel: pour calculer la matrice de corrélation d'une data frame pandas: 
data.corr()

**Réponse:**

Il existe une relation linéaire négative claire entre le pourcentage de personnes allant au travail à vélo et le taux de maladies cardiaques. Cette relation est moins prononcée avec le tabagisme.

Les variables tabagisme et vélo pour se rendre au travail semblent indépendantes.

Aucun point atypique ou regroupement particulier n’est apparent. Dans l’ensemble, les données semblent visiblement synthétiques ou générées artificiellement.

## Régression linéaire simple avec Numpy

### Exercice - Méthode des moindres carrés ordinaire

**(OLS - Ordinary Least Squares)**

**cf. cours d'Algèbre Linéaire - Chapitre 13 de S. Friedli**

In [ ]:
N: int = len(data)

X: NDArray[np.float64] = data[['biking']].to_numpy()
y: NDArray[np.float64] = data['heart.disease'].to_numpy()

In [ ]:
# Remarquez les dimensions de X (2D) et de y (1D).
# Pouvez-vous expliquer pourquoi ? (astuce : observez le code ci-dessus)

print(f"Forme de X: {X.shape}")
print(f"Forme de y: {y.shape}")

**Solution OLS: $\bf{w}^* = \left( \bf{\tilde{x}}^T \bf{\tilde{x}} \right)^{-1} \bf{\tilde{x}}^T y$**

Utilisez les méthodes numpy pour construire $\bf{\tilde{X}}$ et calculer $\bf{w}^*$.

Réponse à trouver: (w_0, w_1) = (17.70, -0.20)

In [ ]:
X_tilde: NDArray[np.float64] = np.column_stack([np.ones(N), X])

In [ ]:
X_tilde.shape

In [ ]:
w_star: NDArray[np.float64] = np.dot(np.dot(np.linalg.inv(np.dot(X_tilde.T,X_tilde)),X_tilde.T), y)

print(f"(w_0, w_1) = ({w_star[0]:2.2f}, {w_star[1]:2.2f})")

### Visualisons le résultat

Visualiser les données individuelles et le modèle de régression linéaire $\hat{y} = \bf{\tilde{x}}\bf{w}^{*}$

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.scatter(
    x=data['biking'],
    y=data['heart.disease'],
    marker='.',
    label="Mesures",
)
ax.plot(X, X_tilde.dot(w_star), color='red', label="Modèle linéaire")
ax.grid(True)
ax.legend(loc='best', fontsize=10)
ax.set_title("Vélo au travail vs santé cardiaque\nMesures en % de la population dans chaque ville", fontsize=12)
ax.set_xlabel("Vélo au travail (% population)", fontsize=10)
ax.set_ylabel("Maladies cardiaques (% population)", fontsize=10)
ax.tick_params(axis='both', labelsize=10) 
plt.show()

### Calculons l'erreur

In [ ]:
res: NDArray[np.float64] = y - X_tilde.dot(w_star)

**Rappel**
L'erreur devrait être distribuée selon une loi normale (moyenne 0).

Visualisons donc la distribution de l'erreur (normalisée) et superposons la densité de probabilité de loi normale correspondante.

💡 Astuce: utilisez _fit_ et _pdf_ [scypy.stats.norm](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.norm.html) pour estimer, par maximum de vraisemblance, la moyenne et l’écart type d’une loi normale à partir de vos données.


In [ ]:
from scipy.stats import norm

In [ ]:
mean_pdf, std_pdf = norm.fit(res)
x_pdf = np.arange(-4.5, 4.5, 0.05)

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(res, density=True, alpha=0.7, color='skyblue', edgecolor='black', label="Histogramme de densité")
ax.plot(x_pdf, norm.pdf(x_pdf, mean_pdf, std_pdf), color='red', label="Loi normale")
ax.legend(loc='best', fontsize=10)
ax.set_title(r"Distribution de l'erreur $\|\hat{y}-y\|$", fontsize=12)
ax.set_xlabel(r"Erreur $\hat{y}=f(X)$", fontsize=10)
ax.set_ylabel("Densité", fontsize=10)
ax.tick_params(axis='both', labelsize=10)
ax.text(1, 0.23, f"(mean={mean_pdf:2.2f}, std={std_pdf:2.2f})", color='red', fontsize=10)

plt.show()

## Régression linéaire simple avec Scikit Learn


### Exercice

Plutôt que de coder l’équation normale des moindres carrés avec NumPy, il est possible d’utiliser les fonctions prêtes à l’emploi de la bibliothèque Scikit-Learn.

⚠️ attention : Scikit-Learn, réclame explicitement que X soit de dimension 2D, de la forme _(N-mesures, n-characteristiques)_.

💡 Astuce : Utilisez les attributs _intercept__ et _coef__ du model entraîné pour construire le vector $w^*$

In [ ]:
# Vérifier que X est 2D, de forme N mesures de 1 characteristique (vélo au travail)
X.shape

In [ ]:
from sklearn.linear_model import LinearRegression

model_univariate = LinearRegression()

model_univariate.fit(X, y)

In [ ]:
w_star_univariate: NDArray[np.float64] = np.array([[model_univariate.intercept_], [model_univariate.coef_[0]]])

In [ ]:
print(f"(w_0, w_1) = ({w_star_univariate[0][0]:2.2f}, {w_star_univariate[1][0]:2.2f})")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.scatter(
    x=data['biking'],
    y=data['heart.disease'],
    marker='.',
    label="Mesures",
)
ax.plot(X, X_tilde.dot(w_star_univariate), color='red', label="Modèle linéaire")
ax.grid(True)
ax.legend(loc='best', fontsize=10)
ax.set_title("Vélo au travail vs santé cardiaque\nMesures en % de la population dans chaque ville", fontsize=12)
ax.set_xlabel("Vélo au travail (% population)", fontsize=10)
ax.set_ylabel("Maladies cardiaques (% population)", fontsize=10)
ax.tick_params(axis='both', labelsize=10) 
plt.show()

### Evaluation du modèle simple

**Mesure de la qualité de la prédiction d'une régression.**

- Erreur absolue moyenne (MAE)

- Racine de l’erreur quadratique moyenne (RMSE)


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
print("MAE             : ", mean_absolute_error(y, model_univariate.predict(X)))
print("Root MSE (RMSE) : ", np.sqrt(mean_squared_error(y, model_univariate.predict(X))))

## Régression linéaire multiple

### Exercice

Considérons maintenant 2 variables indépendantes: biking et smoking.

Utiliser la regression linéaire de scikit learn pour calculer les paramètres du modèle de régression par la méthode des moindres carrées ordinaire (OLS).

In [ ]:
X_multivariate = data[['biking','smoking']].to_numpy()
y = data['heart.disease'].to_numpy()

In [ ]:
X_multivariate_tilde = np.column_stack([np.ones(N), X_multivariate] )

In [ ]:
model_multivariate = LinearRegression()

model_multivariate.fit(X_multivariate, y)

In [ ]:
w_star_multivariate: NDArray[np.float64] = np.array([
    [model_multivariate.intercept_],
    [model_multivariate.coef_[0]],
    [model_multivariate.coef_[1]]
])

In [ ]:
# Réponse: w_star_multivariate = [[14.98],[-0.20],[0.18]]
print(f"(w_0, w_1, w_2) = ({w_star_multivariate[0][0]:2.2f}, {w_star_multivariate[1][0]:2.2f}, {w_star_multivariate[2][0]:2.2f})")

### Evaluation du modèle multivarié

In [ ]:
print("MAE             :", mean_absolute_error(y, model_multivariate.predict(X_multivariate)))
print("Root MSE (RMSE) :", np.sqrt(mean_squared_error(y, model_multivariate.predict(X_multivariate))))

### Calculons l'erreur

In [ ]:
res_multivariate = y - X_multivariate_tilde.dot(w_star_multivariate).T

⚠️ Attention : les fonctions de tracé 1D de Matplotlib (comme plt.plot) attendent un tableau unidimensionnel. Si l'erreur a la forme (N, 1), vous pouvez la transformer en (N,) avec :

In [ ]:
res_multivariate = res_multivariate.reshape(-1,)

In [ ]:
res_multivariate.ndim

### Visualisons la distribution de l'erreur (normalisée) et superposons la densité de probabilité de loi normale correspondante

Rappel: L'erreur devrait être distribuée selon une loi normale (de moyenne nulle)

In [ ]:
mean_pdf, std_pdf = norm.fit(res_multivariate)
x_pdf = np.arange(-4.5, 4.5, 0.05)

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(x=res_multivariate, density=True, alpha=0.7, edgecolor='black', color='skyblue', label="Histogramme de densité")
ax.plot(x_pdf, norm.pdf(x_pdf, mean_pdf, std_pdf), color='red', label="Loi normale")
ax.legend(loc='best', fontsize=10)
ax.set_title(r"Distribution de l'erreur $\|\hat{y}-y\|$", fontsize=12)
ax.set_xlabel(r"Erreur $\hat{y}=f(X)$", fontsize=10)
ax.set_ylabel("Densité", fontsize=10)
ax.tick_params(axis='both', labelsize=10) 
ax.text(1, 0.5, f"(mean={mean_pdf:2.2f}, std={std_pdf:2.2f})", color='red', fontsize=10)

plt.show()

### Visualisation conjointe des données et du modèle de régression en 3D

In [ ]:
%matplotlib widget
A = np.arange(0, 80, 0.5)
B = np.arange(0, 35, 0.5)
A, B = np.meshgrid(A, B)
Z = w_star_multivariate[0] + w_star_multivariate[1]*A + w_star_multivariate[2]*B

fig = plt.figure(figsize=(8,6))
ax = fig.add_subplot(projection='3d')
ax.view_init(elev=15., azim=45)
ax.plot_wireframe(A, B, Z, alpha=0.3, color='red', label="Modèle")
ax.scatter(data['biking'], data['smoking'], data['heart.disease'], marker='^', color='blue', label="Valeurs réelles")
ax.set_title("Vélo et tabagisme v.s. Santé cardiaque", fontsize=12)
ax.set_xlabel("Vélo au travail (% population)", fontsize=10)
ax.set_zlabel("Maladies cardiaques (% population)", fontsize=10)
ax.set_ylabel("Fumeurs (% of population)", fontsize=10)
ax.tick_params(axis='both', labelsize=10) 
ax.legend(loc='best', fontsize=10)

plt.show()

In [ ]:
plt.close("all")
import plotly.express as px
import plotly.graph_objects as go

A = np.arange(0, 80, 0.5)
B = np.arange(0, 35, 0.5)
A, B = np.meshgrid(A, B)

Z = w_star_multivariate[0] + w_star_multivariate[1]*A + w_star_multivariate[2]*B

fig = go.Figure(data = [go.Scatter3d(x=data['biking'], y=data['smoking'], z=data['heart.disease'], mode ='markers', marker_size=4),
                        go.Surface(x=A, y=B, z=Z)])

fig.update_layout(
    width=900,   # in pixels
    height=700,  # in pixels
    title="Vélo et tabagisme v.s. Santé cardiaque",
    scene=dict(
        xaxis_title="Vélo au travail (%)",
        yaxis_title="Fumeurs (%)",
        zaxis_title="Maladies cardiaques (%)"
    )
)

fig.show()

## Régression linéaire comme modèle de prédiction

### Quel % de maladie cardiaque pour {biking = 30%, smoking = 5.5%}?

Jusqu’à présent, nous avons vu comment estimer les paramètres d’un modèle linéaire afin de minimiser l’erreur entre les observations et les valeurs prédites.

Nous allons maintenant utiliser ce modèle pour faire des prédictions sur de nouvelles données : par exemple, quel serait le taux de maladies cardiaques pour une ville où 30 % de la population fait du vélo et 5,5 % fume ?

In [ ]:
X_pred: NDArray[np.float64] = np.array([[30.0, 5.5]])

prediction = model_multivariate.predict(X_pred)

print(">>> Prédiction maladie cardiaque: {prediction[0]:2.1f}%")

### Divisons les données entre un jeu pour l'apprentissage du modèle et un pour l'évaluation du modèle

Jusqu’ici, nous avons uniquement mesuré l’erreur d’apprentissage, celle que le modèle cherche à minimiser pendant l’ajustement.

Cette erreur nous renseigne sur la qualité de l’ajustement des données connues, mais pas sur la capacité du modèle à prédire correctement de nouvelles observations.

Pour évaluer les performances réelles d’un modèle de prédiction, il est essentiel de séparer les données en deux parties :

- un **jeu d’apprentissage** (training set), utilisé pour estimer les paramètres du modèle ;

- un **jeu de test** (test set), réservé à l’évaluation de sa capacité de généralisation.

C’est cette approche, **systématique en apprentissage automatique supervisé**, que nous allons maintenant mettre en œuvre.

In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(X_multivariate, y, test_size=0.2)

In [ ]:
y_train.shape

### Exercice

**Apprenons le modèle de régression multivariate sur le jeu "_train"**

In [ ]:
model_multivariate = LinearRegression()

model_multivariate.fit(x_train, y_train)

**Et évaluons-le sur le jeu "_test" (MAE et RMSE)**

In [ ]:
y_test_hat: NDArray[np.float64] = model_multivariate.predict(x_test)

In [ ]:
y_test_hat.shape

In [ ]:
print("MAE             : ", mean_absolute_error(y_test, y_test_hat))
print("Root MSE (RMSE) : ", np.sqrt(mean_squared_error(y_test, y_test_hat)))

**Enfin, visualisons les 2 jeux de données, les prédictions ainsi que le modèle de régression**

In [ ]:
w_star_train: NDArray[np.float64] = np.array([[model_multivariate.intercept_],[model_multivariate.coef_[0]],[model_multivariate.coef_[1]]])

In [ ]:
%matplotlib widget
fig = plt.figure(figsize=(8,6))

A = np.arange(0, 80, 0.5)
B = np.arange(0, 35, 0.5)
A, B = np.meshgrid(A, B)
Z = w_star_train[0] + w_star_train[1]*A + w_star_train[2]*B


ax = fig.add_subplot(projection='3d')
ax.view_init(elev=15., azim=45)
ax.plot_wireframe(A, B, Z, alpha=0.3, color='green', label="Modèle linéaire")
#ax.scatter(x_train[:,0], x_train[:,1], y_train, marker='o', color='blue')
ax.scatter(x_test[:,0], x_test[:,1], y_test, marker='^', color='red', label="Valeurs réelles")
ax.scatter(x_test[:,0], x_test[:,1], y_test_hat, marker='^', color='green', label="Prédiction")
ax.set_title("Vélo et tabagisme v.s. Santé cardiaque", fontsize=12)
ax.set_xlabel("Biking to work (% of population)", fontsize=10)
ax.set_ylabel("Smoking (% of population)", fontsize=10)
ax.set_zlabel("Heart disease (% of population)", fontsize=10)
ax.tick_params(axis='both', labelsize=10) 
ax.legend(loc='best', fontsize=10)

plt.show()